# µJump dashboard: synaptic-connectivity data builder

**What this notebook does.** For each of the ~19 cell types in µJump, this samples up to 40 cells of that type, looks up ALL of their real synapses in MICrONS via CAVE (both the synapses they *receive* and the ones they *make*), and tallies which cell types those synaptic partners belong to. The result is a small summary — "Pericytes' synaptic partners are mostly X, Y, Z" etc — for every cell type. That summary is the data behind the dashboard's "3 cell types with the most synaptic connections" graph.

**Why this runs here, not inside µJump itself.** Querying real synapses needs **your personal CAVE token** (an auth credential), and querying thousands of synapses for hundreds of sampled cells takes a couple of minutes — both are a poor fit for something that runs live in every visitor's browser. So this is a ONE-TIME (or occasional re-run) offline step: run it here, get one small file out (`connectivity_aggregate.json`), then that file gets dropped next to the dashboard, which is instant for every visitor after that. Your CAVE token stays in this Colab session only — it is never written to the output file, and never seen by anyone else.

**How to run this notebook**
1. Run each cell below in order (click the ▶ button on the left of a cell, or select it and press **Shift+Enter**). Wait for each one to finish before moving to the next.
2. The token-setup cell will pause and ask you to paste your CAVE token — see that cell for how to get one if you don't already have one.
3. The upload cell will ask you to upload one file: **`root_id_to_type.json`** (found next to this notebook — about 5 MB). Use the file picker that pops up.
4. The last cell writes `connectivity_aggregate.json` and offers to download it straight to your computer. Send that file back (or drop it next to `ujump_dashboard.html`) once you have it — the dashboard picks it up automatically.

You do not need to understand every line to run this — just run the cells in order.

## Step 1 — install the CAVE client library (MICrONS' official Python API)

In [ ]:
!pip install --quiet caveclient


In [ ]:
import json, time, collections
from caveclient import CAVEclient

DATASTACK = "minnie65_public"
SYNAPSE_TABLE = "synapses_pni_2"


## Step 2 — your CAVE token

Paste your existing CAVE token into the `TOKEN = "..."` line below, between the quotes, then run the cell. It's saved only to this Colab session/machine, never written to any file this notebook produces.

**Important:** `CAVEclient(DATASTACK)` (naming an actual dataset) tries to fetch that dataset's info immediately, which needs a token to already be set up — creating it before a token exists fails with `AuthException`. So this cell first creates a bare client with **no** dataset name just to save the token, and only creates the real `client` for `minnie65_public` afterwards.

In [ ]:
# A client with no datastack name yet -- just for setting up the token.
auth_client = CAVEclient()

TOKEN = "PASTE_YOUR_CAVE_TOKEN_HERE"  # <-- paste your token between the quotes
auth_client.auth.save_token(token=TOKEN, overwrite=True)

# NOW create the real, dataset-specific client -- it will pick up the token you just saved.
client = CAVEclient(DATASTACK)
print("Connected to", DATASTACK)


## Step 3 — upload `root_id_to_type.json`

This is a lookup file (root ID → cell type) generated alongside this notebook, so a synaptic partner's root ID can be classified locally without a second CAVE query per partner. Run the cell below and use the file picker to upload it (it's next to this notebook, about 5 MB).

In [ ]:
from google.colab import files
import os

if "root_id_to_type.json" not in os.listdir("."):
    print("Please upload root_id_to_type.json now ...")
    files.upload()

with open("root_id_to_type.json") as f:
    ROOT_TO_TYPE = json.load(f)
print("Loaded", len(ROOT_TO_TYPE), "known cell identities.")


## Step 4 — which cells to sample per type

Generated alongside this notebook and small enough to paste in directly. Ask Claude to regenerate this cell if the µJump dataset changes later. Feel free to trim these lists (e.g. keep only the first 10 root IDs per type) for a much faster first test run.

In [ ]:
ROOT_IDS_BY_TYPE = {
  "L6 IT pyramidal neuron": [
    "864691135270503845",
    "864691136602141905",
    "864691135889506953",
    "864691135429439024",
    "864691135638997819",
    "864691136134865035",
    "864691135386158165",
    "864691135115389849",
    "864691136436421278",
    "864691135515924691",
    "864691135724220843",
    "864691135408016585",
    "864691135762549302",
    "864691136483794476",
    "864691135399038241",
    "864691135182189314",
    "864691135762611254",
    "864691135492253791",
    "864691135475532864",
    "864691135741630612",
    "864691136329374954",
    "864691136011775139",
    "864691136143377972",
    "864691135476105792",
    "864691135569461382",
    "864691135337740774",
    "864691136935939295",
    "864691136176002822",
    "864691135345908127",
    "864691135841781731",
    "864691136772813166",
    "864691136389107319",
    "864691136716306798",
    "864691135856145966",
    "864691134886791162",
    "864691135212615808",
    "864691137054528630",
    "864691135394029813",
    "864691135884733296",
    "864691136065725336"
  ],
  "L2/3 pyramidal neuron": [
    "864691136318630110",
    "864691135829919404",
    "864691135225746498",
    "864691135987814915",
    "864691135385267797",
    "864691135958014605",
    "864691135383781867",
    "864691135440767378",
    "864691136031699387",
    "864691136065801880",
    "864691135344317963",
    "864691135777465277",
    "864691135371154555",
    "864691135191757035",
    "864691135940045734",
    "864691135728505748",
    "864691135294828381",
    "864691135211934072",
    "864691135233124697",
    "864691135150480889",
    "864691136922746084",
    "864691134939990115",
    "864691136379207381",
    "864691135858491438",
    "864691135969929984",
    "864691135858491438",
    "864691137010913089",
    "864691135092912332",
    "864691135944460541",
    "864691135373442366",
    "864691136020282104",
    "864691136724804861",
    "864691134848709079",
    "864691135584214136",
    "864691135246002110",
    "864691135013912814",
    "864691136723477245",
    "864691135474699072",
    "864691135771595515",
    "864691135761995318"
  ],
  "Pericyte": [
    "864691135102659680",
    "864691136990250133",
    "864691135771604987",
    "864691135647227439",
    "864691136834349166",
    "864691136031718587",
    "864691135012515830",
    "864691136020328952",
    "864691135918774704",
    "864691136871108718",
    "864691136020265208",
    "864691135212415680",
    "864691135945548068",
    "864691135501544002",
    "864691136742667630",
    "864691135799854946",
    "864691135976966979",
    "864691134990358138",
    "864691135322933660",
    "864691135408645321",
    "864691135478274502",
    "864691136272983486",
    "864691135725622591",
    "864691135761553718",
    "864691135739870612",
    "864691136214668266",
    "864691135351114711",
    "864691136951572959",
    "864691136195787240",
    "864691136237465916",
    "864691136723432957",
    "864691135428540208",
    "864691136194371798",
    "864691136913493233",
    "864691135509036041",
    "864691136927518026",
    "864691136296744475",
    "864691135684400823",
    "864691135888522633",
    "864691135613005508"
  ],
  "Astrocyte": [
    "864691134991210618",
    "864691136237337359",
    "864691136100148085",
    "864691135114460313",
    "864691135494007952",
    "864691136118466212",
    "864691136050844403",
    "864691134917415178",
    "864691136991176085",
    "864691136662389470",
    "864691135501839170",
    "864691135181730818",
    "864691135989520387",
    "864691135945373220",
    "864691135748748329",
    "864691135783124275",
    "864691135700286587",
    "864691136137044605",
    "864691135405412590",
    "864691135373866313",
    "864691136041031510",
    "864691135358996056",
    "864691135278216609",
    "864691135338772454",
    "864691136965845710",
    "864691135807290909",
    "864691135504372661",
    "864691135753644237",
    "864691136992282517",
    "864691135275006225",
    "864691136013112739",
    "864691135926299348",
    "864691136276406541",
    "864691135463672382",
    "864691136334882483",
    "864691135520130954",
    "864691135416111546",
    "864691135256469167",
    "864691136370800264",
    "864691136602085329"
  ],
  "L5 IT pyramidal neuron": [
    "864691135473409074",
    "864691136336989662",
    "864691136742470492",
    "864691134964818975",
    "864691136617923547",
    "864691135118847197",
    "864691135100918816",
    "864691135208557177",
    "864691135859067240",
    "864691136002433994",
    "864691135499777811",
    "864691135294667788",
    "864691135101795104",
    "864691135473003058",
    "864691136227934801",
    "864691135684981687",
    "864691135850847943",
    "864691136673639047",
    "864691135383018970",
    "864691135163288109",
    "864691136091675828",
    "864691136522392209",
    "864691136373066376",
    "864691135469044620",
    "864691136595698850",
    "864691136741495388",
    "864691135441035080",
    "864691136058865368",
    "864691135479091142",
    "864691136594980258",
    "864691135697800346",
    "864691136034597051",
    "864691135785689284",
    "864691135431879728",
    "864691135719727665",
    "864691135367849721",
    "864691135772556539",
    "864691135405722862",
    "864691136091895220",
    "864691136195116108"
  ],
  "Basket cell": [
    "864691135472988722",
    "864691136904061106",
    "864691135687703008",
    "864691136335581875",
    "864691135952513827",
    "864691136123792166",
    "864691135518527626",
    "864691136009280430",
    "864691135841748963",
    "864691135463640382",
    "864691135947338337",
    "864691135661349104",
    "864691135618174863",
    "864691136296748827",
    "864691135841756899",
    "864691135348297943",
    "864691135756085453",
    "864691136595495586",
    "864691135539058802",
    "864691135610308615",
    "864691135349067735",
    "864691135495098768",
    "864691136005624778",
    "864691135527713115",
    "864691135762640438",
    "864691135506077636",
    "864691136391158911",
    "864691136194084822",
    "864691136536141218",
    "864691135874401678",
    "864691136134827659",
    "864691135975465795",
    "864691136743347292",
    "864691136379717845",
    "864691135776646496",
    "864691135737788548",
    "864691135657773954",
    "864691135638952507",
    "864691135800195426",
    "864691135135628569"
  ],
  "Bipolar cell": [
    "864691136331421674",
    "864691135994470337",
    "864691135214110264",
    "864691135518618250",
    "864691136743457884",
    "864691136021592568",
    "864691135294151948",
    "864691136237481276",
    "864691135878346579",
    "864691135584738669",
    "864691135490665831",
    "864691135345113247",
    "864691135876155987",
    "864691135702887291",
    "864691135431978800",
    "864691135875989134",
    "864691135339003622",
    "864691135516904659",
    "864691136136647805",
    "864691135115058329",
    "864691135210025408",
    "864691135658703234",
    "864691136453098239",
    "864691135938236676",
    "864691136053225715",
    "864691135585753155",
    "864691135937627652",
    "864691135213708928",
    "864691135817952079",
    "864691136593436066",
    "864691135058810779",
    "864691135928760276",
    "864691135587862652",
    "864691135785096499",
    "864691135873643662",
    "864691135990802944",
    "864691136378756309",
    "864691136137787773",
    "864691135732257721",
    "864691136236744719"
  ],
  "Microglia": [
    "864691135464618909",
    "864691135388514689",
    "864691135462291613",
    "864691136723348221",
    "864691135953969699",
    "864691135976202607",
    "864691135583682936",
    "864691136088955447",
    "864691134947440636",
    "864691135771721547",
    "864691135334524137",
    "864691135976002883",
    "864691136619626715",
    "864691135012381942",
    "864691135809874892",
    "864691135343283121",
    "864691136617131995",
    "864691136056387032",
    "864691135698285466",
    "864691135841483235",
    "864691135012495094",
    "864691135476096936",
    "864691135693150143",
    "864691135774446411",
    "864691135660524016",
    "864691136463655047",
    "864691135772011771",
    "864691135686728631",
    "864691135478302918",
    "864691135440483144",
    "864691135847909214",
    "864691135012466166",
    "864691135362162247",
    "864691135210024640",
    "864691135440457544",
    "864691135233122649",
    "864691135303473831",
    "864691136065280408",
    "864691135687268064",
    "864691134710083886"
  ],
  "Neurogliaform cell": [
    "864691135645344239",
    "864691135307083334",
    "864691135059767963",
    "864691136422829103",
    "864691136362473698",
    "864691136137168779",
    "864691136371600520",
    "864691135163103533",
    "864691135490593127",
    "864691136489637394",
    "864691135764678198",
    "864691136029895134",
    "864691135584771181",
    "864691135609934343",
    "864691136663185630",
    "864691135501850717",
    "864691135518305747",
    "864691136990942101",
    "864691135526405723",
    "864691136066431128",
    "864691136912671473",
    "864691136914081521",
    "864691135661341936",
    "864691135595712299",
    "864691136137926781",
    "864691136291075223",
    "864691136445946755",
    "864691135659498114",
    "864691135429597232",
    "864691134965402911",
    "864691135851376071",
    "864691135741166699",
    "864691135937601540",
    "864691135927179476",
    "864691135458266610",
    "864691135919921328",
    "864691135136552217",
    "864691136143984180",
    "864691135501814365",
    "864691135762585398"
  ],
  "L4 pyramidal neuron": [
    "864691135715209242",
    "864691134885167866",
    "864691136031760315",
    "864691136966113998",
    "864691134940343395",
    "864691135304251559",
    "864691135476935336",
    "864691136968529870",
    "864691136903972786",
    "864691135397314593",
    "864691134989286522",
    "864691135441375304",
    "864691135941120116",
    "864691135783449907",
    "864691135516276947",
    "864691136041560830",
    "864691136578586132",
    "864691136057226968",
    "864691136090940852",
    "864691136065296792",
    "864691135207744633",
    "864691136134811275",
    "864691135375919048",
    "864691135155564900",
    "864691135448157330",
    "864691135292176310",
    "864691136907569518",
    "864691135645673199",
    "864691136274005005",
    "864691136390771063",
    "864691136194942678",
    "864691136065276824",
    "864691136389273719",
    "864691136524295313",
    "864691135013035766",
    "864691135274101221",
    "864691136390391167",
    "864691135066292804",
    "864691136204617406",
    "864691136312236378"
  ],
  "L5 NP pyramidal neuron": [
    "864691136119421208",
    "864691136314548285",
    "864691135756968637",
    "864691136065299864",
    "864691135324552604",
    "864691136437589406",
    "864691135865954181",
    "864691135776155053",
    "864691135989647107",
    "864691136618439565",
    "864691136867548526",
    "864691136144364340",
    "864691135994201025",
    "864691136391096191",
    "864691135848843102",
    "864691135463694398",
    "864691135660207234",
    "864691135502604098",
    "864691135156580452",
    "864691135954732808",
    "864691134887138298",
    "864691135463691838",
    "864691135848926302",
    "864691135780790864",
    "864691135567759340",
    "864691135884895344",
    "864691136031786939",
    "864691135494508432",
    "864691135431412272",
    "864691135509582089",
    "864691136089432119",
    "864691135385232725",
    "864691135590891019",
    "864691136143688756",
    "864691136137751933",
    "864691135593604548",
    "864691135384027627",
    "864691135511604432",
    "864691136594897314",
    "864691136175931142"
  ],
  "L5 ET pyramidal neuron": [
    "864691136362324706",
    "864691136578396948",
    "864691136335121331",
    "864691135662059760",
    "864691135473731122",
    "864691135294360844",
    "864691136136630909",
    "864691136912571889",
    "864691135463606846",
    "864691135270550437",
    "864691135473084210",
    "864691135867555862",
    "864691137020535406",
    "864691135754580178",
    "864691136334398387",
    "864691135065009988",
    "864691135409058249",
    "864691137197615681",
    "864691136143415348",
    "864691135784278067",
    "864691136371604872",
    "864691135463255965",
    "864691135515947731",
    "864691135502570845",
    "864691135701358715",
    "864691135686883552",
    "864691135059398811",
    "864691136008759724",
    "864691136966620366",
    "864691135848699230",
    "864691136196929100",
    "864691135492195167",
    "864691136488178706",
    "864691136596965026",
    "864691136672258301",
    "864691135429677872",
    "864691135590554891",
    "864691136423609135",
    "864691136197127756",
    "864691135755912402"
  ],
  "Endothelial cell": [
    "864691135718478897",
    "864691135536557503",
    "864691135536557503",
    "864691136295231895",
    "864691135346267102",
    "864691136112586577",
    "864691136174993670",
    "864691135994812874",
    "864691135585388924",
    "864691135858491438",
    "864691135337903474",
    "864691135303545767",
    "864691134029098875",
    "864691136036504712",
    "864691135376106569",
    "864691135954427608",
    "864691136306265229",
    "864691135147256772",
    "864691136039226786",
    "864691135740018580",
    "864691136682488829",
    "864691135858491438",
    "864691135455264362",
    "864691136613057308",
    "864691135995048849",
    "864691136199850686",
    "864691135553836740",
    "864691135202448675",
    "864691135303545767",
    "864691135680979140",
    "864691135303545767",
    "864691135858491438",
    "864691135843705880",
    "864691135858491438",
    "864691136014669720",
    "864691135858491438",
    "864691136175022086",
    "864691135385648981",
    "864691135865884636",
    "864691134915045498"
  ],
  "Oligodendrocyte": [
    "864691135497637907",
    "864691135639227963",
    "864691135502031682",
    "864691135274675941",
    "864691135100116000",
    "864691136000919167",
    "864691135788094493",
    "864691136031695291",
    "864691135718623537",
    "864691135661790135",
    "864691135386397569",
    "864691136119062692",
    "864691135445566866",
    "864691136617432283",
    "864691135807268637",
    "864691135657828482",
    "864691135081717239",
    "864691135639234619",
    "864691135386969429",
    "864691135940776358",
    "864691136378899413",
    "864691135684508343",
    "864691134887397882",
    "864691135738448788",
    "864691136618702221",
    "864691135700278651",
    "864691135369057273",
    "864691135449838578",
    "864691136452242687",
    "864691135497637907",
    "864691135699285282",
    "864691135194737962",
    "864691135360149319",
    "864691136065802648",
    "864691136033964184",
    "864691136011228323",
    "864691135181834242",
    "864691136959717916",
    "864691135342952389",
    "864691136451478527"
  ],
  "Martinotti cell": [
    "864691135779146080",
    "864691135937979396",
    "864691135777789280",
    "864691136925981258",
    "864691135118247133",
    "864691135203379840",
    "864691135082539767",
    "864691135876957523",
    "864691135737505156",
    "864691135349900503",
    "864691135730115001",
    "864691136279941773",
    "864691135737181572",
    "864691135463680062",
    "864691135385695573",
    "864691135012867734",
    "864691135842156771",
    "864691136286939331",
    "864691135104347981",
    "864691136237445948",
    "864691135374389065",
    "864691135463507525",
    "864691136925353290",
    "864691135936463107",
    "864691135396790561",
    "864691136388575760",
    "864691135122296615",
    "864691136483888172",
    "864691136943778527",
    "864691135361306951",
    "864691135725106603",
    "864691135824619102",
    "864691136578792468",
    "864691135293490956",
    "864691134956132354",
    "864691135511236082",
    "864691136108096985",
    "864691135928528852",
    "864691135257219759",
    "864691135762678838"
  ],
  "OPC": [
    "864691135571468581",
    "864691135901611340",
    "864691136311794237",
    "864691135135026336",
    "864691135440715336",
    "864691135776644960",
    "864691135689555680",
    "864691135847931998",
    "864691135816676943",
    "864691136549902754",
    "864691135851563975",
    "864691135732616377",
    "864691136052142323",
    "864691135800991586",
    "864691135407608265",
    "864691135889893769",
    "864691135464455237",
    "864691135587904124",
    "864691136331610346",
    "864691135464948381",
    "864691135122417959",
    "864691135497653779",
    "864691136273707198",
    "864691135995495594",
    "864691135781207376",
    "864691135489479738",
    "864691136324728215",
    "864691136965761742",
    "864691135345180575",
    "864691135349155543",
    "864691136004710090",
    "864691135558231748",
    "864691135885146992",
    "864691135448810324",
    "864691136194987084",
    "864691136008954542",
    "864691135570699501",
    "864691136071928456",
    "864691135572294693",
    "864691135374922569"
  ],
  "L6 CT pyramidal neuron": [
    "864691136099029635",
    "864691135386391169",
    "864691135755016658",
    "864691136619062157",
    "864691135491546087",
    "864691136577479956",
    "864691136253912542",
    "864691135778121661",
    "864691136618492813",
    "864691136287677379",
    "864691136195721430",
    "864691135947327329",
    "864691135156904804",
    "864691135946412324",
    "864691136911650801",
    "864691135781550928",
    "864691135619456911",
    "864691135585894524",
    "864691136878498158",
    "864691136066319512",
    "864691135938128644",
    "864691135684197362",
    "864691135013264022",
    "864691136521772945",
    "864691135518632074",
    "864691135213941304",
    "864691135755887058",
    "864691135772006139",
    "864691135864976382",
    "864691136041986902",
    "864691136020238840",
    "864691136021136632",
    "864691136444543107",
    "864691135397213985",
    "864691135561541857",
    "864691135816432463",
    "864691135730196665",
    "864691136423192111",
    "864691135885663600",
    "864691136137503101"
  ],
  "Perivascular Fibroblast": [
    "864691135986478216",
    "864691135333848814",
    "864691135700391316",
    "864691135767279396",
    "864691135676920944",
    "864691134956045611",
    "864691135787301052",
    "864691135243579401",
    "864691135835815084",
    "864691135573627378",
    "864691136252839825",
    "864691135631964643",
    "864691135190457691",
    "864691135276694973",
    "864691134999731701",
    "864691135489804018",
    "864691136015483064",
    "864691136264351189",
    "864691135462714942",
    "864691135628697155",
    "864691136316189302",
    "864691135307981631",
    "864691136923051748",
    "864691135918600112",
    "864691136286813379",
    "864691135547323933",
    "864691134118086353",
    "864691136239032124",
    "864691135320234850",
    "864691135319949757",
    "864691135812791751",
    "864691136750947438",
    "864691134787608825",
    "864691135609459207",
    "864691135419348200",
    "864691136903567026",
    "864691135323623880",
    "864691134934629188",
    "864691136745138550",
    "864691135209093720"
  ],
  "Macrophage": [
    "864691136618479245",
    "864691135945898788",
    "864691135181795330",
    "864691135182073602",
    "864691135488487994",
    "864691135918600112",
    "864691136277580050",
    "864691135782413875",
    "864691135644706031",
    "864691136903567026",
    "864691135856009518",
    "864691136070942600",
    "864691136438496670",
    "864691136202177726",
    "864691135146947314",
    "864691135272165393",
    "864691135968943973",
    "864691135113516354",
    "864691135581578861",
    "864691135445571218",
    "864691135357405074",
    "864691136903412658",
    "864691136273008318",
    "864691135740465812",
    "864691135122290471",
    "864691135939285761",
    "864691135725600831",
    "864691136124045094",
    "864691135975442755",
    "864691135478301382",
    "864691136902998450",
    "864691135654112450",
    "864691134675281665",
    "864691135736895364",
    "864691135583649656",
    "864691135785658564",
    "864691135654112450",
    "864691135181790466",
    "864691136521868433",
    "864691135282251344"
  ],
  "Lymphocyte": [
    "864691135625515150",
    "864691136008393644",
    "864691135787192508"
  ]
}


## Step 5 — query real synapses for each sampled cell, batched per type

For each cell type, one call asks CAVE for every synapse where any of that type's sampled cells is the **presynaptic** (sending) side, and a second call asks for the **postsynaptic** (receiving) side — batching all ~40 sampled root IDs into one call each, rather than one call per cell, is much faster and gentler on CAVE's servers.

**Note:** this has not been run against the live CAVE server while writing this notebook (Claude's own sandbox cannot reach CAVE's API), so if `synapse_query`'s exact argument names have changed since, check the current signature with `help(client.materialize.synapse_query)` and adjust `pre_ids=`/`post_ids=` below to match. The [MICrONS tutorial notebooks](https://tutorial.microns-explorer.org) are the reference if anything here errors.

In [ ]:
# ---- Retry helper (added 2026-08-02) -----------------------------------------------------------
# The shared public CAVE server occasionally returns "503 Service Temporarily Unavailable" under
# load -- this is a SERVER-SIDE issue (nginx itself refusing the request, not an auth/query
# problem on our end), so a client-side fix can't prevent it, but it's usually transient and a
# few spaced-out retries often get through. Every synapse_query call below goes through this.
def query_with_retries(fn, *args, max_attempts=4, base_delay=8, **kwargs):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            msg = str(e)
            transient = ("503" in msg or "502" in msg or "504" in msg or "timeout" in msg.lower()
                         or "Temporarily Unavailable" in msg or "ConnectionError" in type(e).__name__)
            if not transient or attempt == max_attempts - 1:
                raise
            wait = base_delay * (2 ** attempt)  # 8s, 16s, 32s by default
            print(f"    transient server error ({msg.splitlines()[0][:120]}) -- retrying in {wait}s "
                  f"(attempt {attempt + 2}/{max_attempts})...")
            time.sleep(wait)
    raise last_err  # pragma: no cover -- loop always returns or raises above

connectivity = {}   # source type -> Counter of partner type -> synapse count
partner_cells = {}  # source type -> Counter of partner type -> DISTINCT partner cell count
synapse_io = {}     # source type -> {input/output synapse totals + per-sampled-cell averages}
failed_types = []   # types that failed even after retries -- re-attempted once more at the end

def query_one_type(src_type, root_ids):
    """Returns True on success (fills connectivity/partner_cells/synapse_io for src_type),
    False if it failed even after retries (caller decides whether to note it as failed)."""
    root_ids_int = [int(r) for r in root_ids]
    syn_counter = collections.Counter()
    seen_partners = collections.defaultdict(set)

    try:
        out_df = query_with_retries(client.materialize.synapse_query, pre_ids=root_ids_int)   # synapses THIS type sends
        time.sleep(0.5)  # brief pause between the two queries for the same type
        in_df = query_with_retries(client.materialize.synapse_query, post_ids=root_ids_int)    # synapses THIS type receives
    except Exception as e:
        print("  Query failed for", src_type, "after retries --", e)
        return False

    for _, row in out_df.iterrows():
        partner = str(row["post_pt_root_id"])
        ptype = ROOT_TO_TYPE.get(partner)
        if ptype:
            syn_counter[ptype] += 1
            seen_partners[ptype].add(partner)

    for _, row in in_df.iterrows():
        partner = str(row["pre_pt_root_id"])
        ptype = ROOT_TO_TYPE.get(partner)
        if ptype:
            syn_counter[ptype] += 1
            seen_partners[ptype].add(partner)

    connectivity[src_type] = dict(syn_counter.most_common(10))
    partner_cells[src_type] = {k: len(v) for k, v in seen_partners.items()}
    # 2026-08-02: input/output synapse counts per type, added for the dashboard's new
    # "input vs output synapses per cell type" chart -- out_df/in_df are already fetched above
    # for the partner-type tally, so this just re-uses them rather than querying again.
    # len(root_ids_int) (not len(out_df)/len(in_df)) is the denominator for the per-cell
    # averages, since it's the actual sample size regardless of how many synapses turned up.
    n_sampled = len(root_ids_int)
    synapse_io[src_type] = {
        "n_sampled_cells": n_sampled,
        "output_synapses_total": int(len(out_df)),
        "input_synapses_total": int(len(in_df)),
        "avg_output_synapses_per_cell": round(len(out_df) / n_sampled, 1) if n_sampled else 0,
        "avg_input_synapses_per_cell": round(len(in_df) / n_sampled, 1) if n_sampled else 0,
    }
    return True

# ---- First pass over every type ----
for src_type, root_ids in ROOT_IDS_BY_TYPE.items():
    print("Querying synapses for", src_type, "(", len(root_ids), "sampled cells )...")
    if not query_one_type(src_type, root_ids):
        failed_types.append(src_type)
    time.sleep(1.5)  # be polite to the shared CAVE server between types (raised from 0.5s
                      # on 2026-08-02 after a run hit repeated 503s -- gentler pacing plus the
                      # retry helper above should make transient server load less disruptive)

# ---- Second pass: one more attempt at whatever failed, after a longer cool-down ----
# If the WHOLE first pass failed (e.g. the server was down for a few minutes), retrying
# immediately would likely just fail again -- wait longer once, then try the failed types again.
if failed_types:
    print(f"\n{len(failed_types)} type(s) failed in the first pass ({', '.join(failed_types)}) "
          f"-- waiting 90s then trying them once more...")
    time.sleep(90)
    still_failed = []
    for src_type in failed_types:
        print("Retrying", src_type, "...")
        if not query_one_type(src_type, ROOT_IDS_BY_TYPE[src_type]):
            still_failed.append(src_type)
        time.sleep(1.5)
    failed_types = still_failed

print(f"\nDone querying. {len(ROOT_IDS_BY_TYPE) - len(failed_types)} of {len(ROOT_IDS_BY_TYPE)} "
      f"types succeeded." + (f" Still failing after retries: {', '.join(failed_types)} -- "
      f"if this list isn't empty, the CAVE server may be down for longer than a couple of "
      f"minutes; wait a while and re-run just this cell (Steps 1-4 don't need to be redone in "
      f"the same Colab session)." if failed_types else ""))


## Step 6 — save the result and download it

In [ ]:
out = {
    "note": "Synaptic partner-type tallies, sampled up to 40 cells per source type "
            "(fewer for rare types). by_synapse_count sums individual synapses; "
            "by_distinct_partner_cells counts unique partner cells once each regardless "
            "of how many synapses they share with the sampled cells. synapse_io (added "
            "2026-08-02) gives per-type input/output synapse totals AND per-sampled-cell "
            "averages for the same sample -- the average is the fairer cross-type comparison, "
            "since different cell types have very different sample sizes here.",
    "sample_size_per_type": {t: len(v) for t, v in ROOT_IDS_BY_TYPE.items()},
    "by_synapse_count": connectivity,
    "by_distinct_partner_cells": partner_cells,
    "synapse_io": synapse_io,
}
with open("connectivity_aggregate.json", "w") as f:
    json.dump(out, f, indent=2)
print("Wrote connectivity_aggregate.json")

try:
    from google.colab import files
    files.download("connectivity_aggregate.json")
except Exception:
    print("Download prompt unavailable -- find connectivity_aggregate.json in the Files panel on the left and download it by hand (right-click -> Download).")
